# A bone cell atlas

Four hospitals pooled single-cell RNA-seq from human bone and bone marrow: **1,000 cells from 16
donors**. Nobody has told us which cell is which. The consortium wants three things:

1. **Which cell types are in bone, and how are they related to each other?**
2. Facility D reports a group of cells that no other facility has. **Is it a new cell type?**
3. **Does osteoporosis shift the balance between the cells that dissolve bone and the cells that build
   it?**

Every session so far came with the answer column: a label `y` the model had to learn. Today there is
none. This is **unsupervised learning**: we ask the data for its structure, and then we have to decide
what that structure means.

## Bone in one picture

Bone is a living tissue. It is a hard, mineralised matrix with the **marrow** inside it, and it is
rebuilt all the time:

- **Osteoblasts** build new bone. Some of them get walled into the bone they made and become
  **osteocytes**, which sense mechanical load.
- **Osteoclasts** dissolve old bone. They are made from blood monocytes, so they are blood cells by
  origin, even though they work on bone.
- **Chondrocytes** make the cartilage that covers the joint ends.
- The marrow holds the **blood-forming cells**: stem and progenitor cells (HSPC), and the myeloid,
  lymphoid and red-cell lines they produce. It also holds their **support cells**, the CAR cells
  (*CXCL12-abundant reticular cells*: net-like cells that make the signal CXCL12, which keeps blood stem
  cells in place), and the **vessels**: arteries and wide, thin-walled sinusoids.
- In a healthy adult, building and dissolving are in balance. In **osteoporosis** the balance tips
  towards dissolving, and the bone becomes thin and fragile.

![Bone cells and where they live](https://raw.githubusercontent.com/nbrg-ppcu/appliedmedtech/main/notebooks/session_07/img/bone_cells.png)

*The cells are coloured by what they do. Which genes they use is for you to find out.*

## How this works

There are 11 tasks in 6 parts. Every new method is first shown in an **Example** cell: you just run
it and read what comes out. A **Task** then asks you to do something similar, one step per cell; where a
step uses a small new helper (for example a table of clusters), its comment says what it shows. The code
is already there; the line marked `# <-` tells you what to change (a setting, a gene, a method). Run the
cell and answer the question in words. Where you see **Your conclusion**, write one sentence.

**Cells build on each other.** Run them in order: later cells reuse results of earlier ones, such as the
principal components `pca_coordinates` or the map `umap_coordinates`. Where a task needs a result from
an earlier part, the line **Uses:** says which.

**The toolkit** - all the functions you use - is in the file `toolkit.py`, which the setup downloads
next to this notebook. You do not need to open it; `help(plot_dimred)` (or any other function name)
prints what a function does and what its settings mean.

**In Google Colab:** use a Python CPU runtime and run the setup cells first. The data load directly
from GitHub. Use **File → Save a copy in Drive** to keep your work.

[Lab handout](https://github.com/nbrg-ppcu/appliedmedtech/blob/main/notebooks/session_07/S7_lab_handout.md) · [Data dictionary](https://github.com/nbrg-ppcu/appliedmedtech/blob/main/notebooks/session_07/data/bone_DICTIONARY.md) ·
[toolkit.py](https://github.com/nbrg-ppcu/appliedmedtech/blob/main/notebooks/session_07/toolkit.py)

## Setup

Run the next two cells. The first installs the packages (about a minute in Colab); the second downloads
the toolkit and loads the data.

In [ ]:
%pip install -q numpy pandas scipy scikit-learn umap-learn matplotlib igraph

In [ ]:
import os
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"     # show every result in a cell, not only the last one
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", None)          # never cut long text (gene lists) short

BASE = "https://raw.githubusercontent.com/nbrg-ppcu/appliedmedtech/main/notebooks/session_07/"
if not os.path.exists("toolkit.py"):                 # in Colab: put the toolkit next to the notebook
    urllib.request.urlretrieve(BASE + "toolkit.py", "toolkit.py")
from toolkit import *                                # all toolkit functions, e.g. compute_pca(), plot_dimred()

cells = load_bone_cells(data_dir=BASE + "data/")     # 1,000 cells: metadata and 62 genes, no cell types
print(len(cells), "cells,", len(GENES), "genes")

A first look at the table. Each row is one cell: first who it came from and how it was measured, then
62 genes. `n_counts` is the number of RNA molecules counted in the cell (*UMIs*, unique molecular
identifiers); `pct_mito` is the share of them from mitochondrial genes, which rises in damaged or
stressed cells. The gene values are on a log scale, the same for every gene (see the data dictionary).

In [ ]:
cells.head()

## The toolkit at a glance

![The toolkit in one picture](https://raw.githubusercontent.com/nbrg-ppcu/appliedmedtech/main/notebooks/session_07/img/toolkit_overview.png)

Each part below starts with a closer picture of the functions it uses. In the pictures, a solid box is a
function you call (its inputs after `in:`), a pill is what it gives back, and a dashed box is an internal
helper of the toolkit that you never call yourself.

| Family | Functions | What they do |
|---|---|---|
| looking at genes | `plot_two_genes(cells, gene_x, gene_y, color)` | two genes against each other |
| dimension reduction | `compute_pca(cells, n_components)`, `compute_pcoa(cells, distance)`, `compute_nmds(cells, distance)`, `compute_umap(input_coordinates, n_neighbors, seed)` | new coordinates for the cells |
| plotting them | `plot_dimred(method, coordinates, cells, color, ...)` | **one function for all four**: it calls the right plot for `method` |
| reading PCA | `plot_explained_variance(pca_coordinates)`, `compute_pca_loadings(pca_coordinates, component)` | how much each component carries, and which genes make it |
| hierarchical clustering | `build_cluster_tree(data, linkage_method)` → `plot_cluster_tree(cluster_tree, n_clusters)` → `cut_cluster_tree(cluster_tree, n_clusters)` | build the tree, draw it, cut it into clusters |
| other clustering | `cluster_hdbscan(data, ...)`, `cluster_leiden(data, resolution)` | density-based and graph-based clusters |
| describing clusters | `find_marker_genes(cells, clusters)`, `cluster_composition(cells, clusters, by_column)`, `cluster_quality(cells, clusters)`, `compare_clusterings(row_clusters, column_clusters)`, `MARKER_GENES` | what a cluster is, where it comes from |
| scoring | `mean_silhouette(data, clusters)`, `plot_silhouette_by_cluster_count(data, cluster_tree)`; with `expert_labels = reveal_expert_labels(data_dir, cells)`: `score_against_experts(clusters, expert_labels, level)`, `count_cell_types_per_donor(cells, expert_labels, cell_types)` | how good a clustering is |

Every function gets what it works on through its arguments: `cells` is the table loaded above, `data` is
a table of numbers with one row per cell (for clustering: the gene values `cells[GENES]`). `color`
accepts a metadata column (`"facility"`) or gene (`"PTPRC"`) of `cells`, or a clustering (one label per
cell); in `plot_dimred` also a list of names, which draws one panel each.

## A · No labels

Before any method: what is in the table, and what could create structure in it?

In [ ]:
# EXAMPLE - just run this cell and read the output
# Who are the donors? One row per facility, one column per condition; each number counts cells.
pd.crosstab(index=cells["facility"], columns=cells["condition"])

In [ ]:
# EXAMPLE - just run this cell and read the output
# Two genes against each other: each dot is one cell, coloured by its facility
plot_two_genes(cells=cells, gene_x="PTPRC", gene_y="COL1A1", color="facility")

### Task 1 · What is in the table?

1. Make the same table with `handling` instead of `condition`. Which facility handled its tissue
   differently?
2. Draw two other genes against each other, for example `CDH5` (vessels) and `PTPRC` (blood cells).
3. Count how many such two-gene pictures you would need to see every pair of the 62 genes.

**Which of these could make cells *look different* for a reason that has nothing to do with their cell
type?**

In [ ]:
# TASK 1.1 - apply the hint marked '# <-', then run
pd.crosstab(index=cells["facility"], columns=cells["condition"])     # <- columns=cells["handling"]

In [ ]:
# TASK 1.2 - apply the hint marked '# <-', then run
plot_two_genes(cells=cells, gene_x="PTPRC", gene_y="COL1A1", color="facility")   # <- gene_x="CDH5", gene_y="PTPRC" (or two genes of your choice)

In [ ]:
# TASK 1.3 - just run this cell and read the output
number_of_genes = len(GENES)
number_of_gene_pairs = number_of_genes * (number_of_genes - 1) / 2
number_of_gene_pairs

**Your conclusion, in 1 sentence:** what could make cells look different, apart from their type?

## B · Make it visible

1,891 scatter plots is too many. We need a few new axes that carry most of the information: that is
**dimension reduction**. Each method keeps something different: PCA the variation, PCoA the distances,
NMDS the order of the distances, UMAP the neighbourhoods. All four are plotted with the same function,
`plot_dimred(method=..., coordinates=..., color=...)`.

![The dimension reduction functions](https://raw.githubusercontent.com/nbrg-ppcu/appliedmedtech/main/notebooks/session_07/img/toolkit_dimred.png)

### Example · PCA, one step at a time

`compute_pca` computes principal components: new axes, each a weighted mix of the genes, ordered by how
much of the variation between cells they carry.

In [ ]:
# EXAMPLE - just run this cell and read the output
pca_coordinates = compute_pca(cells=cells, n_components=20)          # 20 principal components of the 62 genes
pca_coordinates.head()                                  # one row per cell, one column per component

How much of the variation does each component carry?

In [ ]:
# EXAMPLE - just run this cell and read the output
plot_explained_variance(pca_coordinates=pca_coordinates)

PC1 against PC2, each dot a cell, coloured by the blood-cell gene `PTPRC`:

In [ ]:
# EXAMPLE - just run this cell and read the output
plot_dimred(method="pca", cells=cells, coordinates=pca_coordinates, color="PTPRC")

What is PC1 made of? `compute_pca_loadings` gives the weight of every gene on one component, sorted
from the largest positive to the most negative weight.

In [ ]:
# EXAMPLE - just run this cell and read the output
pc1_loadings = compute_pca_loadings(pca_coordinates=pca_coordinates, component=1)
print("pc1_loadings holds", len(pc1_loadings), "genes, sorted from the largest to the most negative weight")

### Task 2 · PCA: what do the components mean?

1. Look at the genes with the largest weights on PC1: the first rows of `pc1_loadings`. Then look at
   the other end: the last rows.
2. Colour PC1 against PC2 by another gene, `COL1A1`. What kind of cells sit at each end of PC1?
3. From now on, `plot_dimred` can draw the loadings for you, next to the map: `show_loadings=True` adds
   the top genes of both components shown. Run it for PC1 and PC2 coloured by `facility`. What kind of
   cells does PC2 separate?
4. **Which component carries the facility?** Change the axes to PC3 and PC4, then to PC5 and PC6. Along
   which component does one facility stand apart, and what kind of genes does it weigh most (look them
   up in the data dictionary)?

In [ ]:
# TASK 2.1 - apply the hint marked '# <-', then run
pc1_loadings.head(n=8)                                  # <- then change .head to .tail to see the other end

In [ ]:
# TASK 2.2 - apply the hint marked '# <-', then run
plot_dimred(method="pca", cells=cells, coordinates=pca_coordinates, color="PTPRC")   # <- color="COL1A1"

In [ ]:
# TASK 2.3 - just run this cell and read the output
plot_dimred(method="pca", cells=cells, coordinates=pca_coordinates, color="facility",
            x_axis="PC1", y_axis="PC2", show_loadings=True)

In [ ]:
# TASK 2.4 - apply the hint marked '# <-', then run
plot_dimred(method="pca", cells=cells, coordinates=pca_coordinates, color="facility",
            x_axis="PC1", y_axis="PC2", show_loadings=True)              # <- x_axis="PC3", y_axis="PC4"; then "PC5" and "PC6"

### Example · PCoA

PCoA starts from the **distance** between every pair of cells and finds coordinates whose distances
match them. With the straight-line (Euclidean) distance it should give the same map as PCA, up to the
sign of the axes (the map may come out mirrored).

In [ ]:
# EXAMPLE - just run this cell and read the output
pcoa_euclidean = compute_pcoa(cells=cells, distance="euclidean")
plot_dimred(method="pcoa", cells=cells, coordinates=pcoa_euclidean, color="PTPRC")

### Task 3 · PCoA and NMDS with other distances

The distance is a choice. The toolkit knows four (`help(compute_distance_matrix)` explains them):

- `"euclidean"`: straight-line distance over the genes;
- `"correlation"`: only the pattern of high and low genes counts, not the level;
- `"braycurtis"`: the share of the total amount that differs - the microbiome distance of Session 3;
- `"aitchison"`: for compositions (shares of a whole): compares the *ratios* between genes.

NMDS keeps only the *order* of the distances; its title shows the *stress* (0 perfect, below 0.2
usable). It is computed the same way as PCoA, with `compute_nmds(cells=cells, distance=...)`, and plotted with
`method="nmds"`. Each NMDS takes about half a minute.

1. Run PCoA with the **correlation** distance.
2. Run NMDS with the **Euclidean** distance.
3. Run NMDS with the **Bray-Curtis** distance.
4. Run NMDS with the **Aitchison** distance.

Does the picture change between the distances? Which groups of cells stay apart in every map? Two
things to explain: the **Bray-Curtis** map looks very different - which genes have the largest amounts,
and so dominate a distance computed on amounts (step 3.5 shows them)? And the **Aitchison** map looks
almost like the Euclidean one - why might that be, if our values are already log values per 10,000
molecules?

In [ ]:
# TASK 3.1 - apply the hint marked '# <-', then run
pcoa_correlation = compute_pcoa(cells=cells, distance="euclidean")                   # <- distance="correlation"
plot_dimred(method="pcoa", cells=cells, coordinates=pcoa_correlation, color="PTPRC")

In [ ]:
# TASK 3.2 - just run this cell and read the output
nmds_euclidean = compute_nmds(cells=cells, distance="euclidean")
plot_dimred(method="nmds", cells=cells, coordinates=nmds_euclidean, color="PTPRC")

In [ ]:
# TASK 3.3 - apply the hint marked '# <-', then run
nmds_braycurtis = compute_nmds(cells=cells, distance="euclidean")                    # <- distance="braycurtis"
plot_dimred(method="nmds", cells=cells, coordinates=nmds_braycurtis, color="PTPRC")

In [ ]:
# TASK 3.4 - apply the hint marked '# <-', then run
nmds_aitchison = compute_nmds(cells=cells, distance="euclidean")                     # <- distance="aitchison"
plot_dimred(method="nmds", cells=cells, coordinates=nmds_aitchison, color="PTPRC")

In [ ]:
# TASK 3.5 - just run this cell and read the output
mean_amount_per_gene = np.expm1(cells[GENES]).mean()
mean_amount_per_gene.nlargest(n=5)                                      # the 5 genes with the largest amounts

### Example · UMAP

UMAP links each cell to its nearest neighbours and draws a map that keeps those neighbourhoods
together. As is standard for single-cell data, it works on the principal components
(`input_coordinates=pca_coordinates`). Groups of dots on the map are called **islands**. We keep this map
as `umap_coordinates` and use it in later parts to *look at* clusters.

In [ ]:
# EXAMPLE - just run this cell and read the output
umap_coordinates = compute_umap(input_coordinates=pca_coordinates, n_neighbors=15, seed=0)
plot_dimred(method="umap", cells=cells, coordinates=umap_coordinates, color="PTPRC")

### Task 4 · UMAP: what stays, what changes?

1. Colour the map by several metadata columns at once: `plot_dimred` draws one panel per name in a list.
   Add `age`, `n_counts` and `pct_mito` to the list. Which colourings make islands, and which only shade
   the inside of an island?
2. `plot_dimred` can also **compute** a new map: leave out `coordinates` and give what UMAP should work
   on (`input_coordinates=pca_coordinates`) and its settings instead. Make a map with `n_neighbors=5`,
   then one with `n_neighbors=50`.
3. Make a map with another random start, `seed=1`.

What stays the same in every version, and what changes? Can you trust the **size** of an island, or
the **distance between** two islands?

In [ ]:
# TASK 4.1 - apply the hint marked '# <-', then run
plot_dimred(method="umap", cells=cells, coordinates=umap_coordinates,
            color=["facility", "donor_id"])                             # <- add "age", "n_counts", "pct_mito"

In [ ]:
# TASK 4.2 - apply the hint marked '# <-', then run
plot_dimred(method="umap", cells=cells, color="PTPRC", input_coordinates=pca_coordinates, n_neighbors=15)               # <- n_neighbors=5, then n_neighbors=50

In [ ]:
# TASK 4.3 - apply the hint marked '# <-', then run
plot_dimred(method="umap", cells=cells, color="PTPRC", input_coordinates=pca_coordinates, seed=0)                       # <- seed=1

**Your conclusion, in 1 sentence:** what does each of the four methods keep, and which map would you
use to look for cell types?

## C · The tree

Now we group the cells. **Clustering is a separate method from dimension reduction**: it works on the
62 gene values themselves, not on a map. We use the UMAP map from part B only to *look at* the clusters.
(Whether clustering on a reduced version of the data helps is a question of its own, for later.)

Hierarchical clustering gives not one grouping but a whole **tree** of them:
cut it high and you get a few large groups, cut it low and you get many small ones. Three functions
work together: `build_cluster_tree()` builds it, `plot_cluster_tree()` draws it, `cut_cluster_tree()`
turns it into clusters. The picture also shows the two other ways to get clusters, used in part D, and
the functions that describe any clustering.

![The clustering functions](https://raw.githubusercontent.com/nbrg-ppcu/appliedmedtech/main/notebooks/session_07/img/toolkit_clustering.png)

### Example · build the tree, cut it, read the clusters

First the input: the gene values, one row per cell. **Ward** linkage then merges, at every step, the two
groups whose merge makes the spread inside the groups grow the least, so it builds compact groups.

In [ ]:
# EXAMPLE - just run this cell and read the output
gene_values = cells[GENES]                              # the 62 gene columns: what every clustering works on
ward_tree = build_cluster_tree(data=gene_values, linkage_method="ward")   # the full merge history of the cells
plot_cluster_tree(cluster_tree=ward_tree, n_clusters=3)       # the top of the tree, cut into 3 clusters

Cut the tree into 3 clusters and show them on the UMAP map:

In [ ]:
# EXAMPLE - just run this cell and read the output
ward_3_clusters = cut_cluster_tree(cluster_tree=ward_tree, n_clusters=3)
plot_dimred(method="umap", cells=cells, coordinates=umap_coordinates, color=ward_3_clusters)

What makes each cluster special? Its top genes:

In [ ]:
# EXAMPLE - just run this cell and read the output
find_marker_genes(cells=cells, clusters=ward_3_clusters)

### Task 5 · Does the linkage matter?

1. Build the tree with `"average"` linkage instead (`help(build_cluster_tree)` explains the linkages),
   cut it into 3 clusters and look at the map.
2. Do the same with `"single"` linkage.
3. Count the cells in each of the single-linkage clusters.

What goes wrong with single linkage?

*Uses:* `umap_coordinates` (part B), `gene_values` (the example above).

In [ ]:
# TASK 5.1 - apply the hint marked '# <-', then run
average_tree = build_cluster_tree(data=gene_values, linkage_method="ward")                # <- linkage_method="average"
average_3_clusters = cut_cluster_tree(cluster_tree=average_tree, n_clusters=3)
plot_dimred(method="umap", cells=cells, coordinates=umap_coordinates, color=average_3_clusters)

In [ ]:
# TASK 5.2 - apply the hint marked '# <-', then run
single_tree = build_cluster_tree(data=gene_values, linkage_method="ward")                 # <- linkage_method="single"
single_3_clusters = cut_cluster_tree(cluster_tree=single_tree, n_clusters=3)
plot_dimred(method="umap", cells=cells, coordinates=umap_coordinates, color=single_3_clusters)

In [ ]:
# TASK 5.3 - just run this cell and read the output
single_3_clusters.value_counts()

### Example · name the clusters

Cut the Ward tree much lower, into 16 clusters, and read their top genes. To name a cluster, compare its
top genes with `MARKER_GENES`, the marker table of the data dictionary.

In [ ]:
# EXAMPLE - just run this cell and read the output
ward_16_clusters = cut_cluster_tree(cluster_tree=ward_tree, n_clusters=16)
find_marker_genes(cells=cells, clusters=ward_16_clusters, n_top_genes=6)

In [ ]:
# EXAMPLE - just run this cell and read the output
MARKER_GENES                                            # cell type -> the genes to look for

Names are kept in a *dictionary*: cluster number → name. Every cluster starts as `"?"`, and one is
filled in as an example: cluster 4 has COL1A1, RUNX2, BGLAP and SPP1 on top, which `MARKER_GENES` lists
for the osteoblast.

In [ ]:
# EXAMPLE - just run this cell and read the output
cluster_names = {cluster_number: "?" for cluster_number in range(1, 17)}
cluster_names[4] = "osteoblast"                         # COL1A1, RUNX2, BGLAP, SPP1
plot_dimred(method="umap", cells=cells, coordinates=umap_coordinates, color=ward_16_clusters.map(cluster_names))

### Task 6 · Name the clusters, and read the tree

1. Give every cluster a name, the same way. Clusters you cannot name stay `"?"`. If a cluster's top
   genes fit two cell types, name it with both (for example `"B cell + T cell"`): what does that say
   about the cut?
2. Draw the tree cut into 7 clusters.
3. `compare_clusterings` shows which of your named clusters fall into which of the 7 branches.
   **Where do the osteoclasts sit?** They work on bone - are they on the same branch as the
   bone-forming cells?

In [ ]:
# TASK 6.1 - apply the hint marked '# <-', then run
cluster_names[1] = "?"                                  # <- replace each "?" with your name (cluster 4 is already named)
cluster_names[2] = "?"
cluster_names[3] = "?"
cluster_names[5] = "?"
cluster_names[6] = "?"
cluster_names[7] = "?"
cluster_names[8] = "?"
cluster_names[9] = "?"
cluster_names[10] = "?"
cluster_names[11] = "?"
cluster_names[12] = "?"
cluster_names[13] = "?"
cluster_names[14] = "?"
cluster_names[15] = "?"
cluster_names[16] = "?"
plot_dimred(method="umap", cells=cells, coordinates=umap_coordinates, color=ward_16_clusters.map(cluster_names))

In [ ]:
# TASK 6.2 - apply the hint marked '# <-', then run
plot_cluster_tree(cluster_tree=ward_tree, n_clusters=3)                 # <- n_clusters=7

In [ ]:
# TASK 6.3 - just run this cell and read the output
ward_7_clusters = cut_cluster_tree(cluster_tree=ward_tree, n_clusters=7)
compare_clusterings(row_clusters=ward_7_clusters, column_clusters=ward_16_clusters.map(cluster_names))

**Your conclusion, in 1 sentence:** what does the tree say about how the cell types are related?

## D · Density and graphs

Ward always gives you exactly the number of clusters you ask for, and puts every cell somewhere.
HDBSCAN does neither: it finds the dense regions itself and is allowed to say "this cell belongs
nowhere" (noise, label `-1`). Graph clustering (Leiden: the default of scanpy; Seurat uses the
closely related Louvain) sits in between: no fixed number of clusters, but every cell in one.

### Example · HDBSCAN

`cluster_hdbscan` looks for crowded regions among the cells, using the gene values. `min_cluster_size` is the
smallest group it will call a cluster; `selection_mode="eom"` keeps the groups that stay stable over the
widest range of densities. Noise cells are drawn in grey.

In [ ]:
# EXAMPLE - just run this cell and read the output
hdbscan_clusters = cluster_hdbscan(data=gene_values, min_cluster_size=15, selection_mode="eom")
plot_dimred(method="umap", cells=cells, coordinates=umap_coordinates, color=hdbscan_clusters)

What are these clusters? Their top genes:

In [ ]:
# EXAMPLE - just run this cell and read the output
find_marker_genes(cells=cells, clusters=hdbscan_clusters)

### Task 7 · Finer clusters, and a score without labels

1. Run HDBSCAN with `selection_mode="leaf"`, which keeps the finest groups. What changed, and how many
   cells are now noise?
2. Keep `"leaf"` and change `min_cluster_size` to 5, then to 40. How do the number of clusters and the
   noise change?
3. Lay the leaf clusters (`min_cluster_size=15`) next to your 16 Ward clusters. Where do they agree?
   (Row `-1` holds the noise cells.)

Then a score that needs no labels - first the example, then step 4.

*Uses:* `umap_coordinates` (part B), `gene_values`, `ward_tree`, `ward_16_clusters` (part C).

In [ ]:
# TASK 7.1 - apply the hint marked '# <-', then run
hdbscan_leaf_clusters = cluster_hdbscan(data=gene_values, min_cluster_size=15, selection_mode="eom")  # <- selection_mode="leaf"
plot_dimred(method="umap", cells=cells, coordinates=umap_coordinates, color=hdbscan_leaf_clusters)

In [ ]:
# TASK 7.2 - apply the hint marked '# <-', then run
hdbscan_resized_clusters = cluster_hdbscan(data=gene_values, min_cluster_size=15, selection_mode="leaf")  # <- min_cluster_size=5, then min_cluster_size=40
plot_dimred(method="umap", cells=cells, coordinates=umap_coordinates, color=hdbscan_resized_clusters)

In [ ]:
# TASK 7.3 - just run this cell and read the output
compare_clusterings(row_clusters=hdbscan_leaf_clusters, column_clusters=ward_16_clusters)

### Example · the silhouette

The silhouette scores a clustering without any labels. For one cell it compares *a*, the mean distance
to the other cells of its own cluster, with *b*, the mean distance to the cells of the nearest other
cluster: (b - a) / max(a, b). Near 1: the cell sits deep inside its cluster; near 0: on a border;
negative: it would fit better next door. The mean over all cells scores the whole clustering.

`plot_silhouette_by_cluster_count` cuts the Ward tree into 2, 3, ... 20 clusters and scores each cut.

In [ ]:
# EXAMPLE - just run this cell and read the output
plot_silhouette_by_cluster_count(data=gene_values, cluster_tree=ward_tree)

4. Which number of clusters does the silhouette like best? Now compare three clusterings with
   `mean_silhouette`: your 16 Ward clusters, HDBSCAN (eom) and HDBSCAN (leaf). Noise cells are left out
   of the score, so HDBSCAN is scored on fewer cells than Ward (fewer still for leaf). Is the
   clustering with the highest silhouette the most useful one for an atlas?

In [ ]:
# TASK 7.4 - just run this cell and read the output
pd.Series({"Ward, 16 clusters": mean_silhouette(data=gene_values, clusters=ward_16_clusters),
           "HDBSCAN eom": mean_silhouette(data=gene_values, clusters=hdbscan_clusters),
           "HDBSCAN leaf": mean_silhouette(data=gene_values, clusters=hdbscan_leaf_clusters)})

### Example · Leiden

`cluster_leiden` links every cell to its 15 nearest other cells (by their gene values), which makes a *graph*, and splits the graph
into *communities*: groups with many links inside and few between. `resolution` sets how finely it
splits.

In [ ]:
# EXAMPLE - just run this cell and read the output
leiden_clusters = cluster_leiden(data=gene_values, resolution=1.0)
plot_dimred(method="umap", cells=cells, coordinates=umap_coordinates, color=leiden_clusters)

### Task 8 · Leiden at other resolutions

1. Run Leiden at resolution **0.3**, then at **2**. How many clusters does each give?
2. Lay the resolution-1 clusters next to your Ward clusters. Do the two methods find the same groups?
3. Where do the resolution-1 clusters come from: is there one that comes from a single facility only?

Which would you choose for the atlas - Ward, HDBSCAN or Leiden - and at which setting? You do not have
to be sure yet: part F will tell you more.

*Uses:* `umap_coordinates` (part B), `gene_values`, `ward_16_clusters` (part C).

In [ ]:
# TASK 8.1 - apply the hint marked '# <-', then run
leiden_other_resolution = cluster_leiden(data=gene_values, resolution=1.0)                # <- resolution=0.3, then resolution=2
plot_dimred(method="umap", cells=cells, coordinates=umap_coordinates, color=leiden_other_resolution)

In [ ]:
# TASK 8.2 - just run this cell and read the output
compare_clusterings(row_clusters=leiden_clusters, column_clusters=ward_16_clusters)

In [ ]:
# TASK 8.3 - just run this cell and read the output
cluster_composition(cells=cells, clusters=leiden_clusters, by_column="facility")

### Task 9 · Who is noise?

HDBSCAN (leaf) called many cells noise. Noise holds two kinds of cells: cells at the **edge** of a
cluster, where the density thins out, and odd cells that fit **nowhere**. The cells below do the
looking for you, one step each:

1. Label every cell "noise" or "in a cluster" and draw that on the map.
2. Compare the quality of the two groups: median RNA molecules (`n_counts`), genes and mitochondrial
   share. The median alone hides the odd cells.
3. So sort: the 10 noise cells with the most RNA molecules, with a few marker genes of different cell
   types. **Add two marker genes of your choice** to the list. What is odd about these cells? (Read
   *Doublets* in the data dictionary.)
4. Colour the map by three genes of the bone-forming cells, along their path from progenitor to
   osteocyte. Is there a natural place to split them into types?

*Uses:* `hdbscan_leaf_clusters` (Task 7), `umap_coordinates` (part B).

In [ ]:
# TASK 9.1 - just run this cell and read the output
is_noise = hdbscan_leaf_clusters == -1
noise_label = is_noise.map({True: "noise", False: "in a cluster"})
plot_dimred(method="umap", cells=cells, coordinates=umap_coordinates, color=noise_label)

In [ ]:
# TASK 9.2 - just run this cell and read the output
cluster_quality(cells=cells, clusters=noise_label)

In [ ]:
# TASK 9.3 - apply the hint marked '# <-', then run
columns_to_show = ["n_counts", "PTPRC", "CD3E", "CD14", "HBB", "COL1A1", "PECAM1"]   # <- add two marker genes
noise_cells = cells[is_noise]
noise_cells.nlargest(n=10, columns="n_counts")[columns_to_show]

In [ ]:
# TASK 9.4 - just run this cell and read the output
plot_dimred(method="umap", cells=cells, coordinates=umap_coordinates,
            color=["ALPL", "BGLAP", "SOST"])

**Your conclusion, in 1 sentence:** what does a noise label tell you, and what does it not?

## E · Is every cluster a cell type?

Facility D says it found a cell population that nobody else has. Find it, and decide.

### Task 10 · Facility D's population

1. Where do the cells of each HDBSCAN cluster come from? Is there a cluster whose cells nearly all come
   from **one** facility? Then change `hdbscan_clusters` to `leiden_clusters`: does the same cluster
   appear?
2. What are that cluster's top genes? Look them up in the data dictionary, and read how facility D
   handled its tissue.
3. What does its quality look like (`n_counts`, `pct_mito`)?
4. The map coloured by facility, two stress genes and the mitochondrial share. Apart from this cluster,
   how strongly do the facilities differ?

Is it a new cell type?

*Uses:* `hdbscan_clusters`, `leiden_clusters` (part D), `umap_coordinates` (part B).

In [ ]:
# TASK 10.1 - apply the hint marked '# <-', then run
cluster_composition(cells=cells, clusters=hdbscan_clusters, by_column="facility")    # <- then clusters=leiden_clusters

In [ ]:
# TASK 10.2 - just run this cell and read the output
find_marker_genes(cells=cells, clusters=hdbscan_clusters)

In [ ]:
# TASK 10.3 - just run this cell and read the output
cluster_quality(cells=cells, clusters=hdbscan_clusters)

In [ ]:
# TASK 10.4 - just run this cell and read the output
plot_dimred(method="umap", cells=cells, coordinates=umap_coordinates,
            color=["facility", "FOS", "HSPA1A", "pct_mito"], n_columns=2)

*Is facility D's population a new cell type? Give your reasons below.*

*Your answer:*

**Your conclusion, in 1 sentence:** how can you tell a cell type from an artefact?

## F · The reveal

Only now: the expert panel's labels. Remember that they were made with the whole transcriptome and
the tissue sections, so they know more than our 62 genes.

![The scoring functions](https://raw.githubusercontent.com/nbrg-ppcu/appliedmedtech/main/notebooks/session_07/img/toolkit_scoring.png)

### Example · score one clustering

`reveal_expert_labels()` loads the labels at three levels: `compartment` (3 groups), `lineage` (7) and
`cell_type` (16); at every level the 24 doublets carry one more label, `doublet`. `score_against_experts` scores a clustering against one level with the *adjusted Rand
index* (ARI): 1 means the same cells are together as in the expert labels, 0 means no better than chance.
The names of your clusters do not matter. It gives three numbers: the ARI over all cells (noise counted
as one more cluster), the ARI without the noise cells, and how many cells were kept for the second.

In [ ]:
# EXAMPLE - just run this cell and read the output
expert_labels = reveal_expert_labels(data_dir=BASE + "data/", cells=cells)
score_against_experts(clusters=ward_3_clusters, expert_labels=expert_labels, level="compartment")     # 3 Ward clusters vs 3 compartments

### Task 11 · Score your clusterings, and answer the osteoporosis question

1. Score the 7 Ward clusters against the level they should match.
2. Score your 16 Ward clusters against the level they should match.
3. Score HDBSCAN (leaf) against `cell_type`, then Leiden. For HDBSCAN read both ARIs, with and without
   the noise cells. Which clustering is closest? Did the silhouette of Task 7 rank Ward and HDBSCAN the
   same way as the ARI does?
4. Your names next to the expert labels: which did you get right, and which got mixed up?
5. **The osteoporosis question.** For each donor: the number of osteoclasts and osteoblasts, and the
   osteoclast share.
6. The share summarised by condition. How many independent samples do you have for each condition? And
   look at which facility each condition comes from: could the difference come from something other
   than the disease?

*Uses:* `ward_7_clusters`, `ward_16_clusters`, `cluster_names` (part C), `hdbscan_leaf_clusters`,
`leiden_clusters` (part D).

In [ ]:
# TASK 11.1 - apply the hint marked '# <-', then run
score_against_experts(clusters=ward_7_clusters, expert_labels=expert_labels, level="compartment")    # <- set level to "compartment", "lineage" or "cell_type": which fits 7 clusters?

In [ ]:
# TASK 11.2 - apply the hint marked '# <-', then run
score_against_experts(clusters=ward_16_clusters, expert_labels=expert_labels, level="compartment")   # <- set level to "compartment", "lineage" or "cell_type": which fits 16 clusters?

In [ ]:
# TASK 11.3 - apply the hint marked '# <-', then run
score_against_experts(clusters=hdbscan_leaf_clusters, expert_labels=expert_labels, level="cell_type")   # <- then clusters=leiden_clusters

In [ ]:
# TASK 11.4 - just run this cell and read the output
compare_clusterings(row_clusters=ward_16_clusters.map(cluster_names),
                    column_clusters=expert_labels["cell_type"])

In [ ]:
# TASK 11.5 - just run this cell and read the output
counts_per_donor = count_cell_types_per_donor(cells=cells, expert_labels=expert_labels, cell_types=["osteoclast", "osteoblast"])
counts_per_donor["osteoclast share"] = (counts_per_donor["osteoclast"]
                                        / (counts_per_donor["osteoclast"] + counts_per_donor["osteoblast"])).round(2)
counts_per_donor

In [ ]:
# TASK 11.6 - just run this cell and read the output
counts_per_donor.groupby(by="condition")["osteoclast share"].agg(["mean", "min", "max", "count"])

**Your conclusion, in 1 sentence:** what did the reveal show about your clusters?

## The report

Three lines, as you would write them in a thesis, then **upload this notebook to Moodle**.

**The atlas I would publish, at which level of the tree, and with which method:**

**Facility D's population:**

**Does osteoporosis shift the balance, how many samples support the answer, and what else could explain it:**

In [ ]:
# scratch: space for your own code